# Notebook for Adding the Simplified Interventions Conditions

This is a simple notebook that takes the simplified (plain language) interventions and conditions in the dataset and combines them to the data in `extracted_interventions_conditions.json`.

The simplified versions of interventions and conditions are derived from the model-extracted ones. 
First, Qwen3-4B-Instruct-2507 was used to initially simplify the terms. Then, the researchers with the help of health communication and medical experts manually reviewed and improved the terms. 

The goal was to create terms that would most likely be used by people with low health literacy levels while being clinically accurate.

In [ ]:
import pandas as pd

In [ ]:
simplified_df = pd.read_csv("simplified_interventions_conditions.csv")

In [ ]:
# filter out rows where the "excluded" column is 1
simplified_df = simplified_df[simplified_df["excluded"] != 1]

In [ ]:
extracted_df = pd.read_json("extracted_interventions_conditions.json")

In [ ]:
# add the "simplified_intervention" and "simplified_condition" columns from simplified csv to the extracted_interventions_conditions dataframe using "ReviewID" column as key
# the key to save is "SimplifiedExtractedText" which is a combination of "simplified_intervention" and "simplified_condition" in json format: {"intervention": ..., "condition": ...}
simplified_dict = simplified_df.set_index("ReviewID")[["simplified_intervention", "simplified_condition"]].to_dict(orient="index")
extracted_df["SimplifiedExtractedText"] = extracted_df["ReviewID"].map(
    lambda x: {
        "intervention": simplified_dict[x]["simplified_intervention"], 
        "condition": simplified_dict[x]["simplified_condition"]
    } if x in simplified_dict else {"intervention": None, "condition": None}
)

In [ ]:
# save the resulting dataframe
extracted_df.to_json("extracted_interventions_conditions_new.json", orient="records")